<h1 align="center">
  <b>RAG-Benchmark-Lab Series 1</b>
</h1>
<h2>
  <b>Advanced Document Parsing for RAG Systems</b>
</h2>

Estimated time needed: **45** minutes

## __Table of Contents__

<ol>
    <li><a href="#Overview">Overview</a></li>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Mounting-Google-Drive">Mounting Google Drive & API Keys</a></li>
        </ol>
    </li>
    <li>
        <a href="#Document-Parsers-Evaluation">Document Parsers Evaluation</a>
        <ol>
            <li><a href="#PyMuPDF">PyMuPDF (Standard Extraction)</a></li>
            <li><a href="#LlamaParse">LlamaParse (Agentic Extraction)</a></li>
            <li><a href="#Docling">Docling (Deep Structural Extraction)</a></li>
            <li><a href="#LandingAI">Landing.AI DPT (Vision-Based Extraction)</a></li>
        </ol>
    </li>
    <li><a href="#Comparison">Time Performance Comparison</a></li>
    <li><a href="#Conclusion">Conclusion & Recommendation</a></li>
</ol>


## Overview
<a id="Overview"></a>
When building a Retrieval-Augmented Generation (RAG) pipeline utilizing LangChain or LangGraph, the quality of the document extraction phase directly dictates the performance of the system. While standard parsers can extract plain text, complex documents containing intricate structures—such as the IMF World Economic Outlook report—require advanced parsing techniques to preserve tables, charts, and hierarchical headings.

This laboratory session benchmarks four distinct parsing methodologies — PyMuPDF, LlamaParse, Docling, and Landing.AI DPT — evaluating them based on execution latency (via `time.perf_counter()`) and markdown structural fidelity.

## Objectives
<a id="Objectives"></a>
After completing this lab, you will be able to:
- Implement and benchmark PyMuPDF, LlamaParse, Docling, and Landing.AI DPT within a Python environment.
- Evaluate the trade-offs between execution speed and extraction quality (structural fidelity for tables, headings, and layout).
- Automatically route and persist parsed markdown outputs to Google Drive for subsequent integration with interactive learning platforms like Google NotebookLM.


----
## Setup
<a id="Setup"></a>

### Installing required libraries
The following required libraries are not preinstalled in the Google Colab environment. Run the following cell to install them.

In [ ]:
# Install the necessary parsing libraries
!pip install -q PyMuPDF
!pip install -q llama-parse
!pip install -q docling
!pip install -q python-dotenv
!pip install -q landingai-ade
!pip install -q requests


### Mounting Google Drive & API Keys
<a id="Mounting-Google-Drive"></a>
This notebook works in **both Google Colab and local Jupyter**. The cell below automatically detects which environment it's running in and adjusts accordingly:

- **In Google Colab**: Google Drive is mounted for output storage, and API keys are read from Colab's Secrets manager (🔑 icon in the left sidebar).
- **In local Jupyter**: outputs are saved to a local folder instead, and API keys are read from a `.env` file in the same directory as this notebook (using `python-dotenv`, already installed in the Setup step above).

**Before running the cell below, create API keys for the two cloud-based parsers used in this notebook:**

- **LlamaParse (LlamaCloud):** Sign in at [cloud.llamaindex.ai](https://cloud.llamaindex.ai), open **API Key** in the left sidebar, and click **Generate New Key**. The free tier currently allows up to 1,000 pages/day.
- **Landing.AI DPT (Agentic Document Extraction):** Sign in at [va.landing.ai](https://va.landing.ai) to generate your `VISION_AGENT_API_KEY`. A free trial is available and does not require a credit card.

**If running in Google Colab:** open Colab's **Secrets manager** (the 🔑 icon in the left sidebar), add two secrets named `LLAMA_PARSE_API_KEY` and `DPT_APIKEY`, and make sure **Notebook access** is toggled on for both.

**If running in local Jupyter:** create a file named `.env` in the same folder as this notebook, with the following content:
```
LLAMA_PARSE_API_KEY=your-llamaparse-key-here
DPT_APIKEY=your-landingai-key-here
```


In [ ]:
import time
import os

# --- Detect environment: Google Colab vs local Jupyter ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in {'Google Colab' if IN_COLAB else 'local Jupyter'}")

if IN_COLAB:
    from google.colab import drive, userdata, files

    # Mount Google Drive
    drive.mount('/content/drive')

    # Generic output directory in the root of My Drive
    # (no reference to any specific personal project folder)
    output_dir = '/content/drive/MyDrive/Document_Parsing_Lab_Outputs'
else:
    # Load variables from a local .env file (python-dotenv, installed in Setup)
    from dotenv import load_dotenv
    load_dotenv()

    # Generic output directory relative to this notebook
    output_dir = './Document_Parsing_Lab_Outputs'

os.makedirs(output_dir, exist_ok=True)
print(f"✅ Output directory ready at: {output_dir}")


def get_secret(key_name: str) -> str:
    """Fetch an API key from Colab Secrets (in Colab) or from environment
    variables loaded via a local .env file (outside Colab)."""
    value = userdata.get(key_name) if IN_COLAB else os.environ.get(key_name)
    if not value:
        hint = (
            "Add it via Colab's Secrets manager (🔑 icon)."
            if IN_COLAB
            else f"Add it to a local .env file, e.g. {key_name}=your-key-here"
        )
        raise ValueError(f"Secret '{key_name}' not found. {hint}")
    return value.strip()


# Load API Key for LlamaParse securely (see instructions above to create one)
LLAMA_CLOUD_API_KEY = get_secret('LLAMA_PARSE_API_KEY')
os.environ["LLAMA_PARSE_API_KEY"] = LLAMA_CLOUD_API_KEY

# --- Define the source document dynamically ---
if IN_COLAB:
    # In Colab, upload a PDF directly from your computer.
    # If your PDF is already in Google Drive instead, comment this block out
    # and set document_path manually, e.g.:
    #   document_path = '/content/drive/MyDrive/<your_folder>/<your_file>.pdf'
    uploaded = files.upload()
    document_path = next(iter(uploaded))
else:
    # In local Jupyter, point this to your own PDF, e.g.:
    #   document_path = "/Users/you/Documents/my_report.pdf"
    # By default, this downloads the same benchmark document (IMF WEO Update,
    # July 2026) used throughout this notebook, so it works out of the box.
    document_path = "source_document.pdf"
    if not os.path.exists(document_path):
        import requests
        DOCUMENT_URL = "https://www.imf.org/-/media/files/publications/weo/2026/update/july/english/text.pdf"
        print(f"Downloading source document from {DOCUMENT_URL} ...")
        response = requests.get(DOCUMENT_URL, timeout=60)
        response.raise_for_status()
        with open(document_path, "wb") as f:
            f.write(response.content)

print(f"📄 Using document: {document_path}")


----
## Document Parsers Evaluation
<a id="Document-Parsers-Evaluation"></a>

### 1. PyMuPDF (Standard Extraction)
<a id="PyMuPDF"></a>
PyMuPDF offers exceptional speed for basic text extraction. However, it typically outputs raw strings, losing complex structural elements like HTML tables or nested lists.

> **Why `import fitz` instead of `import pymupdf`?** The PyPI package is named `PyMuPDF`, but its import name is `fitz`. This is a historical naming choice — PyMuPDF was originally built as Python bindings for MuPDF's internal rendering engine, codenamed "Fitz". The package has since added an `import pymupdf` alias in newer versions for clarity, but `import fitz` remains the most common and widely-documented way to use it, so this notebook uses that convention.


In [ ]:
import fitz  # PyMuPDF

print("Starting PyMuPDF extraction...")
start_time = time.perf_counter()

# Open and read the document
doc = fitz.open(document_path)
pymupdf_text = ""
for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    pymupdf_text += page.get_text()

end_time = time.perf_counter()
pymupdf_latency = end_time - start_time

# Save the output to Google Drive
pymupdf_out_path = os.path.join(output_dir, 'PyMuPDF_Output.txt')
with open(pymupdf_out_path, "w", encoding="utf-8") as f:
    f.write(pymupdf_text)

print(f"⏱️ PyMuPDF Execution Time: {pymupdf_latency:.2f} seconds")
print(f"💾 Output saved to: {pymupdf_out_path}")
print("\n--- Snippet ---")
print(pymupdf_text[:500])


### 2. LlamaParse (Agentic Extraction)
<a id="LlamaParse"></a>
LlamaParse is a proprietary, API-based solution designed specifically for RAG applications. It utilizes LLMs to interpret tables and images, transforming them into clean, structured Markdown format.

In [ ]:
from llama_parse import LlamaParse
print("Starting LlamaParse extraction...")

# Retrieve the LLAMA_PARSE_API_KEY (works in both Colab and local Jupyter,
# see the get_secret() helper defined in the Setup cell above)
LLAMA_PARSE_API_KEY = get_secret('LLAMA_PARSE_API_KEY')

start_time = time.perf_counter()

# Initialize parser
parser = LlamaParse(
    api_key=LLAMA_PARSE_API_KEY,
    result_type="markdown",
    verbose=True
)

# Parse document
llama_docs = parser.load_data(document_path)
llama_full_text = "\n\n".join([doc.text for doc in llama_docs])

end_time = time.perf_counter()
llamaparse_latency = end_time - start_time

# Save the output
llama_out_path = os.path.join(output_dir, 'LlamaParse_Output.md')
with open(llama_out_path, "w", encoding="utf-8") as f:
    f.write(llama_full_text)

print(f"⏱️ LlamaParse Execution Time: {llamaparse_latency:.2f} seconds")
print(f"💾 Output saved to: {llama_out_path}")


### 3. Docling (Deep Structural Extraction)
<a id="Docling"></a>
Docling provides robust, open-source structural extraction. While highly accurate at interpreting layouts, it is computationally intensive. When executed on a standard CPU, latency increases significantly compared to API-based alternatives.

In [ ]:
from docling.document_converter import DocumentConverter

print("Starting Docling extraction... (This may take several minutes on CPU)")
start_time = time.perf_counter()

converter = DocumentConverter()
result = converter.convert(document_path)
docling_markdown = result.document.export_to_markdown()

end_time = time.perf_counter()
docling_latency = end_time - start_time

# Save the output to Google Drive
docling_out_path = os.path.join(output_dir, 'Docling_Output.md')
with open(docling_out_path, "w", encoding="utf-8") as f:
    f.write(docling_markdown)

print(f"⏱️ Docling Execution Time: {docling_latency:.2f} seconds")
print(f"💾 Output saved to: {docling_out_path}")


### 4. Landing.AI DPT (Vision-Based Extraction)
<a id="LandingAI"></a>
Landing.AI's Document Parsing Tool (DPT) utilizes advanced vision-based AI models, specifically the `dpt-3-pro` architecture, to interpret complex document layouts. By processing the document as a visual input rather than a mere text stream, it excels at retaining intricate structural formatting within the generated Markdown output.


In [ ]:
from pathlib import Path
from landingai_ade import LandingAIADE

print("Starting Landing.AI DPT extraction...")
start_time = time.perf_counter()

# Retrieve the API key securely (works in both Colab and local Jupyter,
# see the get_secret() helper defined in the Setup cell above)
DPT_APIKEY = get_secret('DPT_APIKEY')

# Initialize the Landing.AI client
client = LandingAIADE(apikey=DPT_APIKEY)

# Execute the document parsing using the dpt-3-pro model
response = client.v2.parse(
    document=Path(document_path),  # We use the document_path variable defined in the Setup phase
    model="dpt-3-pro"
)

end_time = time.perf_counter()
dpt_latency = end_time - start_time

# Extract the markdown content
dpt_markdown = response.markdown

# Save the output
dpt_out_path = os.path.join(output_dir, 'LandingAI_Output.md')
with open(dpt_out_path, "w", encoding="utf-8") as f:
    f.write(dpt_markdown)

print(f"⏱️ Landing.AI Execution Time: {dpt_latency:.2f} seconds")
print(f"💾 Output saved to: {dpt_out_path}")


----
## Time Performance Comparison
<a id="Comparison"></a>
With all four parsers executed, we now consolidate their latency results into a single comparison table and chart. We also add a qualitative assessment of structural fidelity (how well each parser preserves tables, headings, and layout), since raw speed alone doesn't tell the full story for a RAG pipeline.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# NOTE: These variables (pymupdf_latency, llamaparse_latency, docling_latency, dpt_latency)
# were captured live from time.perf_counter() in each parser's cell above.
# Re-running this cell without re-running the extraction cells will reuse the last measured values.
latency_results = {
    "Parser": ["PyMuPDF", "Docling", "LlamaParse", "Landing.AI DPT"],
    "Execution Time (s)": [pymupdf_latency, docling_latency, llamaparse_latency, dpt_latency],
    "Extraction Type": ["Text-only", "Structural (local CPU)", "Agentic (cloud API)", "Vision-based (cloud API)"],
}

df_latency = pd.DataFrame(latency_results).sort_values("Execution Time (s)").reset_index(drop=True)

print("Document Parsing Latency Comparison\n")
display(df_latency.style.format({"Execution Time (s)": "{:.2f}"}).hide(axis="index"))

# --- Visualization ---
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(df_latency["Parser"], df_latency["Execution Time (s)"], color="#4C72B0")
ax.set_ylabel("Execution Time (seconds)")
ax.set_title("Document Parser Latency Comparison")
ax.bar_label(bars, fmt="%.2f")
plt.tight_layout()
plt.show()

# --- Qualitative structural fidelity comparison ---
qualitative_results = {
    "Parser": ["PyMuPDF", "Docling", "LlamaParse", "Landing.AI DPT"],
    "Table Retention": ["Poor (plain text)", "Good", "Very Good", "Very Good"],
    "Heading Hierarchy": ["Lost", "Preserved", "Preserved", "Preserved"],
    "Output Format": ["Plain text", "Markdown", "Markdown", "Markdown"],
    "Cost / Infra": ["Free, local", "Free, local (CPU-heavy)", "Paid API", "Paid API"],
}
df_qualitative = pd.DataFrame(qualitative_results)
print("\nQualitative Structural Fidelity Comparison\n")
display(df_qualitative.style.hide(axis="index"))


----
## Conclusion & Recommendation
<a id="Conclusion"></a>
Based on the latency and structural fidelity results above:

- **PyMuPDF** is the fastest option and is well suited for simple, text-heavy documents where table/layout structure is not important.
- **Docling** offers a strong open-source, locally-run alternative that preserves structure without any API cost, at the expense of higher CPU latency.
- **LlamaParse** and **Landing.AI DPT** both produce high-fidelity, RAG-ready markdown by leveraging cloud-based models, at the cost of API latency and usage fees.

For a production RAG pipeline over structurally complex documents (e.g., financial or policy reports with tables and charts), a hybrid approach is recommended: use PyMuPDF for a fast pre-check, and fall back to Docling or a cloud-based parser (LlamaParse / Landing.AI DPT) when the document contains tables, multi-column layouts, or embedded charts.


<br>
<h3 align="center"><b>About the Author</b></h3>

**Suprapto Santoso** is a developer focused on integrating AI, LangChain, and advanced Retrieval-Augmented Generation (RAG) systems.

This notebook was created as part of a comprehensive evaluation of document parsing methodologies. If you found this laboratory session helpful, you can read the full technical breakdown and methodology in the accompanying article on Medium.

* **Read the article:** https://medium.com/@suprapto-santoso/battle-of-document-parsing-rag-pymupdf-vs-docling-vs-llamaparse-vs-dpt-8d26a2d0175f
* **Connect on LinkedIn:** https://www.linkedin.com/in/suprapto-santoso/
* **View Source Code:** https://github.com/Supra-San/RAG-Benchmark-Lab

<p align="center">
  <i>© 2026 Suprapto Santoso. This work is licensed under a <a href="https://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.</i>
</p>
